# Lab 2: Target Filtering - Dynamic ROI and Point-in-Polygon

Welcome to Lab 2. In this lab, we will build a target filtering mechanism to determine which detected objects are actually dangerous.

---

## Overview

After detecting objects and estimating their positions, the system still cannot directly make control decisions. The key issue is that **not all detected objects lie on the vehicle's trajectory**.

For example:
- A vehicle in another lane → not dangerous  
- An object outside the turning path → not relevant  

👉 Therefore, we define a **Region of Interest (ROI)** that represents the vehicle’s future path.

---

## Core Ideas

This lab consists of two main components:

### 1. Dynamic ROI
- The ROI is not fixed
- It changes according to the steering angle

### 2. Point-in-Polygon (PIP)
- Determines whether an object lies inside the ROI
- Implemented using the Ray Casting algorithm

---

## Learning Objectives

- Construct a dynamic ROI based on heading angle  
- Understand polygon deformation  
- Implement Point-in-Polygon (Ray Casting)  
- Filter relevant (dangerous) objects  

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. Dynamic ROI

## Theory

The ROI is represented as a polygon.

- When the vehicle moves straight → ROI is symmetric  
- When the vehicle turns → ROI shifts laterally  

We simulate this by moving the top vertices of the polygon according to the steering angle.

In [ ]:
def create_dynamic_roi(width, height, heading_deg):
    """
    width, height: BEV dimensions
    heading_deg: steering angle (degrees)
    """

    # Normalize heading (-1 → 1)
    norm = heading_deg / 30.0  # assume max steering = 30 degrees

    # TODO: compute horizontal offset
    offset = norm * width * 0.2

    # Bottom points (fixed)
    bottom_left = (width * 0.3, height)
    bottom_right = (width * 0.7, height)

    # TODO: shift top points using offset
    top_left = (width * 0.3 + offset, height * 0.5)
    top_right = (width * 0.7 + offset, height * 0.5)

    roi = np.array([
        bottom_left,
        bottom_right,
        top_right,
        top_left
    ], dtype=np.float32)

    return roi

## Visualize ROI

In [ ]:
def plot_roi(roi, width, height):
    plt.figure()
    
    xs = list(roi[:,0]) + [roi[0,0]]
    ys = list(roi[:,1]) + [roi[0,1]]
    
    plt.plot(xs, ys, marker='o')
    plt.xlim(0, width)
    plt.ylim(height, 0)
    plt.title("Dynamic ROI")
    plt.show()


roi = create_dynamic_roi(400, 600, heading_deg=15)
plot_roi(roi, 400, 600)

# 2. Point-in-Polygon (Ray Casting)

## Theory

To determine whether a point lies inside a polygon:

- Cast a horizontal ray from the point to infinity  
- Count how many times it intersects polygon edges  

If the number of intersections is **odd** → inside  
If **even** → outside  

In [ ]:
def point_in_polygon(x, y, polygon):
    n = len(polygon)
    inside = False

    px, py = x, y

    for i in range(n):
        x1, y1 = polygon[i]
        x2, y2 = polygon[(i + 1) % n]

        # TODO: check if ray intersects edge
        if ((y1 > py) != (y2 > py)):
            
            # TODO: compute intersection
            x_intersect = x1 + (py - y1) * (x2 - x1) / (y2 - y1)
            
            # TODO: toggle state
            if px < x_intersect:
                inside = not inside

    return inside

# 3. Applying to Object Detection

We use the **bottom-center point** of each bounding box  
to represent the object's position on the ground plane.

In [ ]:
def get_bottom_center(box):
    x1, y1, x2, y2 = box

    # TODO
    cx = (x1 + x2) / 2
    cy = y2

    return cx, cy

In [ ]:
# Simulated bounding boxes
boxes = [
    (150, 200, 200, 300),
    (50, 100, 100, 200),
    (250, 150, 300, 280)
]

roi = create_dynamic_roi(400, 600, heading_deg=10)

print("Objects inside ROI:")

for box in boxes:
    cx, cy = get_bottom_center(box)
    
    inside = point_in_polygon(cx, cy, roi)
    
    print(f"Box {box} -> inside: {inside}")